# 🎙️ All-in-One AI Audio Studio (Gradio Live 터널링 백엔드)
### 코랩 T4 GPU 위에서 23개 언어 더빙(Chatterbox) 및 고정밀 한국어 나레이션(Qwen3) 실행

1. **[Step 1]** 셀을 실행하면 빌드 컴파일러(Cython, build-essential)를 먼저 준비한 후 Chatterbox 및 Qwen3-TTS를 에러 없이 완벽히 설치합니다. (1~2분 소요)
2. **[Step 2]** 셀을 실행하면 모델이 로드되고 Gradio의 공인 터널이 켜지며 **`https://xxxx.gradio.live`** 링크가 발급됩니다!
3. 이 링크를 **GitHub 대시보드**에 붙여넣고 [내장 창으로 열기]를 누르시면 대시보드 안에서 바로 작업하실 수 있습니다.

## 📦 [Step 1] C++ 빌드 환경 구축 및 정식 소스 설치 (무결점 버전)

In [ ]:
# ① 우분투 C/C++ 컴파일 도구 및 사운드 라이브러리 준비
!apt-get update -qq && apt-get install -y -qq build-essential python3-dev libsndfile1-dev git

# ② Cython, wheel 및 packaging 최신화 (spacy-pkuseg 빌드 에러의 근본적 해결)
!pip install -q --upgrade pip "setuptools<81" wheel cython numpy

# ③ spacy-pkuseg 및 resemble-perth 선행 빌드 설치
!pip install -q --no-build-isolation spacy-pkuseg
!pip install -q git+https://github.com/resemble-ai/Perth.git@master

# ④ Chatterbox 공식 소스 클론 및 editable 설치
import os
if not os.path.exists("/content/chatterbox"):
    !git clone https://github.com/resemble-ai/chatterbox.git /content/chatterbox

%cd /content/chatterbox
!pip install -q --no-build-isolation -e .
!pip install -q qwen-tts soundfile gradio

import torch
print("\n" + "="*50)
print("GPU 활성화 확인:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("사용 GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ 상단 메뉴 [런타임] -> [런타임 유형 변경]에서 GPU(T4)를 선택해 주세요!")

# 설치 무결성 검증
try:
    import chatterbox
    from qwen_tts import Qwen3TTSModel
    print("✅ Chatterbox 및 Qwen3-TTS 정식 설치 무결성 검증 완료!")
    print("👉 이제 아래 [Step 2] 셀을 실행하세요.")
    print("="*50)
except Exception as e:
    print("⚠️ 설치 검증 실패:", str(e))


## ⚡ [Step 2] 통합 스튜디오 가동 (공인 Live 링크 자동 발급)
실행 후 나오는 **`Running on public URL: https://xxxx.gradio.live`** 링크를 대시보드에 붙여넣으세요!

In [ ]:
import os, sys, time, subprocess, shutil
import torch, torchaudio, soundfile as sf
import gradio as gr

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 모델 초기화 중... (Device: {DEVICE})")

# 1. Chatterbox Multilingual 로드
from chatterbox.mtl_tts import ChatterboxMultilingualTTS, SUPPORTED_LANGUAGES
cb_model = ChatterboxMultilingualTTS.from_pretrained(device=DEVICE)
print(f"✅ Chatterbox 로드 완료 ({len(SUPPORTED_LANGUAGES)}개 언어 지원)")

# 2. Qwen3-TTS 로드
from qwen_tts import Qwen3TTSModel
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
qwen_model = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-0.6B-Base", device_map="cuda:0" if torch.cuda.is_available() else "cpu", dtype=dtype)
print("✅ Qwen3-TTS 준비 완료")

CACHE_DIR = "/content/audio_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# 단일/배치 통합 합성 함수
def generate_batch_audio(voice_audio, bulk_text, engine, lang, exagg, cfg, ref_text_prompt, progress=gr.Progress()):
    if not voice_audio:
        raise gr.Error("내 목소리 오디오 파일(10초~30초)을 먼저 업로드해 주세요!")
    if not bulk_text or not bulk_text.strip():
        raise gr.Error("생성할 나레이션 텍스트를 입력해 주세요!")
    
    # 1. 목소리 전처리 (24kHz 모노 변환)
    clean_voice = f"{CACHE_DIR}/clean_voice.wav"
    subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", voice_audio, "-ac", "1", "-ar", "24000", "-t", "30", clean_voice], check=True)
    
    # 2. 텍스트 분절 (---, /// 또는 줄바꿈 2번)
    raw_parts = [p.strip() for p in bulk_text.replace('///', '---').split('---') if p.strip()]
    if len(raw_parts) == 1 and '\n\n' in raw_parts[0]:
        raw_parts = [p.strip() for p in raw_parts[0].split('\n\n') if p.strip()]
        
    out_files = []
    zip_dir = f"{CACHE_DIR}/zip_pack_{int(time.time())}"
    os.makedirs(zip_dir, exist_ok=True)
    
    total = len(raw_parts)
    for idx, text in enumerate(raw_parts):
        progress((idx) / total, f"[{idx+1}/{total}] 나레이션 생성 중: {text[:20]}...")
        out_filename = f"narration_{str(idx+1).zfill(2)}.wav"
        out_filepath = os.path.join(zip_dir, out_filename)
        
        if engine == "Qwen3-TTS (한국어 고정밀)":
            q_ref = ref_text_prompt.strip() if ref_text_prompt.strip() else "안녕하세요. 인공지능 목소리 복제 테스트 녹음입니다."
            wavs, sr = qwen_model.generate_voice_clone(
                text=text,
                language="Korean",
                ref_audio=clean_voice,
                ref_text=q_ref
            )
            sf.write(out_filepath, wavs[0], sr)
        else:
            # Chatterbox Multilingual
            wav = cb_model.generate(
                text,
                language_id=lang,
                audio_prompt_path=clean_voice,
                exaggeration=exagg,
                cfg_weight=cfg
            )
            torchaudio.save(out_filepath, wav.cpu(), cb_model.sr)
            
        out_files.append(out_filepath)
        
    # ZIP 파일 생성
    zip_archive_path = f"{CACHE_DIR}/my_narration_pack_{int(time.time())}"
    shutil.make_archive(zip_archive_path, 'zip', zip_dir)
    zip_file = f"{zip_archive_path}.zip"
    
    # 첫 번째 오디오 미리듣기 및 전체 파일 리스트 반환
    first_audio = out_files[0] if out_files else None
    return first_audio, out_files, zip_file, f"총 {total}개의 나레이션 오디오가 성공적으로 생성되었습니다!"

# --- 정돈된 Gradio 스튜디오 UI 구성 ---
custom_theme = gr.themes.Soft(primary_hue="lime", secondary_hue="slate").set(
    body_background_fill="#06080b",
    block_background_fill="#0d1117",
    block_border_color="#21262d",
    body_text_color="#f0f6fc"
)

with gr.Blocks(theme=custom_theme, title="All-in-One AI Audio Studio") as demo:
    gr.Markdown("""
    # 🎙️ All-in-One AI Audio Studio
    ### Connect AI Lab × Resemble AI Chatterbox × Qwen3-TTS
    내 목소리 10초~30초로 **20여 개 나레이션 대본**을 한 번에 털어넣고 순차적으로 일괄 생성합니다.
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            voice_input = gr.Audio(label="🎙️ 내 목소리 샘플 업로드 (10초~30초)", type="filepath")
            engine_choice = gr.Radio(
                choices=["Chatterbox (23개 언어 글로벌 더빙)", "Qwen3-TTS (한국어 고정밀)"],
                value="Chatterbox (23개 언어 글로벌 더빙)",
                label="🎯 합성 엔진 선택"
            )
            lang_choice = gr.Dropdown(
                choices=["en", "ko", "ja", "es", "fr", "zh", "de", "hi", "it", "pt", "ru", "ar"],
                value="en",
                label="🌍 목표 언어 (Chatterbox 전용)"
            )
            with gr.Accordion("⚙️ 정밀 조절 옵션", open=False):
                exagg_slider = gr.Slider(0.25, 1.0, value=0.5, step=0.05, label="감정 세기 (Exaggeration)")
                cfg_slider = gr.Slider(0.1, 1.0, value=0.5, step=0.05, label="말하기 속도/차분함 (CFG Weight)")
                qwen_ref_text = gr.Textbox(label="녹음 대본 (Qwen3 전용)", value="안녕하세요. 인공지능 목소리 복제 테스트 녹음입니다.")
                
        with gr.Column(scale=2):
            text_bulk = gr.Textbox(
                label="📝 20개 내외 영상 나레이션 대본 일괄 입력 (--- 또는 엔터 2번으로 구분)",
                lines=10,
                value="""첫 번째 영상 클립을 소개하는 7초 분량의 오프닝 나레이션입니다.
---
두 번째 장면에서는 실제 인공지능이 어떻게 목소리를 복제하는지 설명합니다.
---
마지막 결론 부에서는 글로벌 23개 언어로 자연스럽게 확장되는 모습을 보여줍니다."""
            )
            btn_submit = gr.Button("🚀 순차 일괄 오디오 생성 시작", variant="primary", size="lg")
            
            status_box = gr.Markdown("대기 중...")
            first_audio_preview = gr.Audio(label="🎧 생성 결과 1번 미리듣기", type="filepath")
            all_audio_files = gr.File(label="📁 생성된 전체 개별 WAV 목록")
            zip_download = gr.File(label="📦 전체 일괄 압축 ZIP 다운로드")
            
    btn_submit.click(
        fn=generate_batch_audio,
        inputs=[voice_input, text_bulk, engine_choice, lang_choice, exagg_slider, cfg_slider, qwen_ref_text],
        outputs=[first_audio_preview, all_audio_files, zip_download, status_box]
    )

# 공식 Gradio Live 공유 링크 발급
demo.queue().launch(share=True, debug=True)
